In [8]:
import numpy as np
import os
import time
from scipy.io import loadmat, savemat

# -----------------------------
# Parameters
# -----------------------------
number = 400
nsx = 32
nsy = 32
nbz = 16

# The density values of the two random starting points were 1 g/cm3 and 0.6 g/cm3
density = np.array([1000.0, 600.0])  
path_m = './model/syn/'
path_d = './data/syn/'
os.makedirs(path_d, exist_ok=True)
# -----------------------------
# Load Gz matrix
# -----------------------------
G_data = loadmat(r'C:\Users\cp dubey\Desktop\Rahul_MTP\Sigle prism cnn\gz_inv\G_matrix.mat')
G = G_data['Gz']  
# -----------------------------
# Main loop
# -----------------------------
start_time = time.time()

for num in range(number):  
    savefile = os.path.join(path_d, f'data{num}.mat')

    d = np.zeros((1024, 1))                  
    con_m = np.zeros((16384, 2))            
    # 16384 = nsx * nsy * nbz
    
    for n_idx, n in enumerate([1, 2]):       # n = 1 or 2, n_idx = 0 or 1
        file = os.path.join(path_m, f'{n}model{num}.dat')  # '1model0.dat', '2model0.dat'

        # MATLAB: input = load(file);
        input_data = np.loadtxt(file)
        if input_data.ndim == 1:
            input_data = input_data.reshape(1, -1)

        modeling_m, modeling_n = input_data.shape

        # MATLAB: m = zeros(nsx * nsy * nbz,1);
        m = np.zeros((nsx * nsy * nbz, 1))
        
        # m(input(i,1) + nsx*(input(i,2)-1) + nsx*nsy*(input(i,3)-1)) = input(i,4)*density(n);
        for i in range(modeling_m):
            ix = int(input_data[i, 0])   # input(i,1)
            iy = int(input_data[i, 1])   # input(i,2)
            iz = int(input_data[i, 2])   # input(i,3)
            val = input_data[i, 3]       # input(i,4)

            index = ix + nsx * (iy - 1) + nsx * nsy * (iz - 1) - 1

            m[index, 0] = val * density[n_idx]   # density(1) or density(2)

        d += G @ m
        
        con_m[:, n_idx] = m[:, 0]

    savemat(savefile, {'d': d, 'con_m': con_m})

end_time = time.time()
print(f"Elapsed time: {end_time - start_time:.2f} seconds")

Elapsed time: 8.32 seconds
